In [2]:
import sys

# Delete the cached module
if 'VaR - domrf' in sys.modules:
    del sys.modules['VaR - domrf']

# Re-import with __import__()
megavar = __import__('VaR - domrf')

import pandas as pd
import jax.numpy as jnp
import numpy as np 
import matplotlib.pyplot as plt
from scipy import stats

In [3]:
returns_student = stats.t.rvs(1, size=5000)
returns_normal = stats.norm.rvs(size=5000)

ES_student = megavar.historical_es(returns_student)
ES_normal = megavar.historical_es(returns_normal)

print(ES_student, ES_normal)

historical_es() took 0.304805s
historical_es() took 0.001996s
2.8756270206505814 0.017137939338365712


In [4]:
#test for jnp.quantile linear interpolation vs indexing approaches
var1: callable = megavar.historical_var

@megavar.Auxiliary.timer
def var2(returns, alpha=0.01): 
    _ = -1 * returns 
    sorted_returns = jnp.sort(_) #lossess are to the right
    index = jnp.floor(alpha * sorted_returns.shape[0]).astype(jnp.int32)    
    var = sorted_returns[index]
    return var 

@megavar.Auxiliary.timer
def var3(returns, alpha=0.01): 
    _ = -1 * returns 
    sorted_returns = np.sort(_) #lossess are to the right
    index = np.floor(alpha * sorted_returns.shape[0]).astype(np.int32)    
    var = sorted_returns[index]
    return var 

In [5]:
t1 = var1(returns_normal)
t2 = var2(returns_normal)
t3 = var3(returns_normal)

historical_var() took 0.090709s
var2() took 0.084749s
var3() took 0.000145s


Parametric ES testing 


In [6]:
#Example 11.10 Hull 
std = 20 
returns_example1110 = stats.norm.rvs(scale=std, size=5000)

t4 = megavar.parametric_es_normal(returns_example1110)
print(t4)

54.01616983084037


GARCH(p, q) ES testing 

In [7]:
t5 = megavar.garch_es(returns_example1110)
print(t5)

garch_es() took 0.032432s
53.9850007432567


EVT ES testing 

In [8]:
returns_student_1 = stats.t.rvs(1, size=5000)
returns_student_2 = stats.t.rvs(2, size=5000)
returns_student_3 = stats.t.rvs(3, size=5000)
returns_student_5 = stats.t.rvs(5, size=5000)

t6 = megavar.EVT_es(returns_student_1)
t7 = megavar.EVT_es(returns_student_2)
t8 = megavar.EVT_es(returns_student_3)
t9 = megavar.EVT_es(returns_student_5)

print(t6,t7,t8,t9)



GPD_ppf() took 0.003130s
MLE_EVT() took 1.019333s
MLE_EVT() took 0.393819s
MLE_EVT() took 0.389427s
MLE_EVT() took 0.465889s
307.0190144971629 74.41913661316532 9.4773863803226 3.1212562374761927


In [9]:
returns_mega = stats.t.rvs(100, size=500)
a = megavar.EVT_es(returns_mega)
print(a)

GPD_ppf() took 0.002924s
MLE_EVT() took 0.567110s
-0.022440185548276623


In [10]:
ess = []
failed = []
for v in range(1, 101): 
    r = stats.t.rvs(v, size=5000)
    model_es = megavar.EVT_es(r)
    ess.append(model_es)
    if model_es < 0:
        print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!', v)
        failed.append(v)



MLE_EVT() took 0.779349s
MLE_EVT() took 1.109424s
MLE_EVT() took 1.021529s
MLE_EVT() took 0.968823s
MLE_EVT() took 1.134367s
MLE_EVT() took 1.069741s
MLE_EVT() took 1.027751s
MLE_EVT() took 1.044470s
MLE_EVT() took 0.989123s
MLE_EVT() took 0.999266s
MLE_EVT() took 1.307389s
MLE_EVT() took 1.013638s
MLE_EVT() took 1.015898s
MLE_EVT() took 1.087619s
MLE_EVT() took 1.221007s
MLE_EVT() took 0.995309s
MLE_EVT() took 1.115000s
MLE_EVT() took 1.085388s
MLE_EVT() took 0.979774s
MLE_EVT() took 1.218171s
MLE_EVT() took 0.989266s
MLE_EVT() took 1.139008s
MLE_EVT() took 0.902158s
MLE_EVT() took 0.918684s
MLE_EVT() took 1.057028s
MLE_EVT() took 1.303613s
MLE_EVT() took 0.973568s
MLE_EVT() took 0.373300s
MLE_EVT() took 0.358478s
MLE_EVT() took 0.356773s
MLE_EVT() took 0.351110s
MLE_EVT() took 0.344450s
MLE_EVT() took 0.354131s
MLE_EVT() took 0.346919s
MLE_EVT() took 0.344407s
MLE_EVT() took 0.344671s
MLE_EVT() took 0.366152s
MLE_EVT() took 0.400724s
MLE_EVT() took 0.432106s
MLE_EVT() took 0.365378s


In [11]:
ess

[Array(298.58418695, dtype=float64),
 Array(73.7066238, dtype=float64),
 Array(8.18094951, dtype=float64),
 Array(7.08598789, dtype=float64),
 Array(5.46535634, dtype=float64),
 Array(4.66097785, dtype=float64),
 Array(1.67049189, dtype=float64),
 Array(1.63026071, dtype=float64),
 Array(0.64056872, dtype=float64),
 Array(0.44721805, dtype=float64),
 Array(0.31988926, dtype=float64),
 Array(4.29910396, dtype=float64),
 Array(2.94210035, dtype=float64),
 Array(4.00404541, dtype=float64),
 Array(3.40552117, dtype=float64),
 Array(0.58960902, dtype=float64),
 Array(0.42181067, dtype=float64),
 Array(0.8040642, dtype=float64),
 Array(0.26027013, dtype=float64),
 Array(2.29665634, dtype=float64),
 Array(2.71647906, dtype=float64),
 Array(1.48306428, dtype=float64),
 Array(1.16964731, dtype=float64),
 Array(2.66422391, dtype=float64),
 Array(1.35478623, dtype=float64),
 Array(1.20291185, dtype=float64),
 Array(1.43208355, dtype=float64),
 Array(2.83030154, dtype=float64),
 Array(1.55238849, 

In [12]:
failed

[57, 64, 67, 84, 86, 92]

In [13]:
#simulatio of the number of times ES is smaller than 0 for a sample of t dist with df = 75 
n = 1000 #number of simulation
failed_75 = 0
for i in range(n): 
    r = stats.t.rvs(75, size=75)
    es_75 = megavar.EVT_es(r)
    if es_75 < 0: 
        failed_75 += 1



GPD_ppf() took 0.002854s
MLE_EVT() took 0.474649s
MLE_EVT() took 0.303282s
MLE_EVT() took 0.318315s
MLE_EVT() took 0.309310s
MLE_EVT() took 0.307758s
MLE_EVT() took 0.316555s
MLE_EVT() took 0.306520s
MLE_EVT() took 0.304409s
MLE_EVT() took 0.319224s
MLE_EVT() took 0.320045s
MLE_EVT() took 0.306506s
MLE_EVT() took 0.304312s
MLE_EVT() took 0.310951s
MLE_EVT() took 0.309756s
MLE_EVT() took 0.311353s
MLE_EVT() took 0.321663s
MLE_EVT() took 0.317683s
MLE_EVT() took 0.310894s
MLE_EVT() took 0.309498s
MLE_EVT() took 0.313615s
MLE_EVT() took 0.308236s
MLE_EVT() took 0.314879s
MLE_EVT() took 0.313403s
MLE_EVT() took 0.482657s
MLE_EVT() took 0.311989s
MLE_EVT() took 0.306911s
MLE_EVT() took 0.313523s
MLE_EVT() took 0.301538s
MLE_EVT() took 0.306048s
MLE_EVT() took 0.315812s
MLE_EVT() took 0.317285s
MLE_EVT() took 0.313359s
MLE_EVT() took 0.318085s
MLE_EVT() took 0.304084s
MLE_EVT() took 0.314845s
MLE_EVT() took 0.308292s
MLE_EVT() took 0.304680s
MLE_EVT() took 0.308337s
MLE_EVT() took 0.313754s


In [14]:
print(failed_75/n)


0.297


Fail rate ~30%

In [24]:
#fail rate for the EVT_var 
n = 100 
df = [1, 2, 3, 4, 5, 7, 10, 15, 30, 50, 75, 100]
fails = {d: [0, [], []] for d in df}
for test in range(n): 
    for v in df: 
        r = stats.t.rvs(v, size=5000)
        var, res, u, params = megavar.EVT_var(r, return_details=True)
        if params[0] < 0 or params[1] < 0: 
            fails[v][0] += 1
            fails[v][1].append(params[0])
            fails[v][2].append(params[1])
# fails[1][0] += 1 
# fails

MLE_EVT() took 0.528190s
IMPORTANT 249 5000 7.210548422646979 0.9437998975023614 0
MLE_EVT() took 0.478851s
IMPORTANT 249 5000 1.1397681493098744 0.8529107832417777 3
MLE_EVT() took 0.479037s
IMPORTANT 249 5000 0.9795322875876342 0.39659383971106 0
MLE_EVT() took 0.486872s
IMPORTANT 249 5000 0.8255254408525119 0.22479366688631916 0
MLE_EVT() took 0.501755s
IMPORTANT 249 5000 0.6771311228828076 0.14443049020431867 0
MLE_EVT() took 0.439495s
IMPORTANT 249 5000 0.9226398269031078 -0.19642394638528926 3
!!!!!!!!!!FLAG
MLE_EVT() took 0.432782s
IMPORTANT 249 5000 0.8375335327650526 -0.6696761698659446 3
!!!!!!!!!!FLAG
MLE_EVT() took 0.425468s
IMPORTANT 249 5000 0.8821608861201575 -0.73434372494566 3
!!!!!!!!!!FLAG
MLE_EVT() took 0.430537s
IMPORTANT 249 5000 0.5659983633097497 -0.08733840892007927 0
!!!!!!!!!!FLAG
MLE_EVT() took 0.428635s
IMPORTANT 249 5000 0.34912716634105406 0.134845609315827 3
MLE_EVT() took 0.430278s
IMPORTANT 249 5000 0.37320298351606307 -0.40813233353294504 3
!!!!!!!!!!

In [26]:
import pickle

with open('fails_EVT_var.pkl', 'wb') as f:
    pickle.dump(fails, f)
fails

{1: [0, [], []],
 2: [0, [], []],
 3: [0, [], []],
 4: [17,
  [Array(1.1473684, dtype=float64),
   Array(1.17125279, dtype=float64),
   Array(1.31575243, dtype=float64),
   Array(1.24906039, dtype=float64),
   Array(1.21641473, dtype=float64),
   Array(1.50680056, dtype=float64),
   Array(1.15047956, dtype=float64),
   Array(1.30061963, dtype=float64),
   Array(1.36161943, dtype=float64),
   Array(1.32011046, dtype=float64),
   Array(1.16100022, dtype=float64),
   Array(1.31667012, dtype=float64),
   Array(1.29645963, dtype=float64),
   Array(1.27936501, dtype=float64),
   Array(1.3342951, dtype=float64),
   Array(1.82004374, dtype=float64),
   Array(1.54504441, dtype=float64)],
  [Array(-0.12640442, dtype=float64),
   Array(-0.24116304, dtype=float64),
   Array(-0.32977977, dtype=float64),
   Array(-0.26543806, dtype=float64),
   Array(-0.1031407, dtype=float64),
   Array(-0.32486259, dtype=float64),
   Array(-0.22998309, dtype=float64),
   Array(-0.23019581, dtype=float64),
   Array(

In [28]:
fails_df = {v: fails[v][0]/100 for v in fails.keys()}
fails_df

{1: 0.0,
 2: 0.0,
 3: 0.0,
 4: 0.17,
 5: 0.21,
 7: 0.82,
 10: 0.96,
 15: 0.83,
 30: 0.75,
 50: 0.68,
 75: 0.74,
 100: 0.81}